# P3.09 Discovery: Network Probe

**PURPOSE:** Validate internet connectivity from Kaggle notebooks

**CRITICAL:** Some features (secrets, external APIs) require network access

**Timeout:** ≤5 minutes

In [ ]:
import socket
import json
import time
from pathlib import Path
from datetime import datetime

results = {
    "session_id": f"network_probe_{int(datetime.now().timestamp())}",
    "timestamp": datetime.now().isoformat(),
    "tests": []
}

def log_test(name, result, evidence, duration_s):
    results["tests"].append({
        "test": name,
        "result": result,
        "evidence": evidence,
        "duration_s": duration_s
    })
    print(f"[{result:8s}] {name}")

start = time.time()
print("🔬 Network Probe Discovery\n")

## Test 1: DNS Resolution

In [ ]:
# Test DNS resolution
test_start = time.time()
try:
    ip = socket.gethostbyname("google.com")
    log_test("DNS Resolution", "PASS", {"domain": "google.com", "ip": ip}, time.time() - test_start)
except Exception as e:
    log_test("DNS Resolution", "FAIL", {"error": str(e)}, time.time() - test_start)

## Test 2: HTTPS Connectivity

In [ ]:
# Test HTTPS connectivity
test_start = time.time()
try:
    import urllib.request
    response = urllib.request.urlopen("https://www.google.com", timeout=5)
    log_test("HTTPS Connectivity", "PASS", {"status": response.status}, time.time() - test_start)
except Exception as e:
    log_test("HTTPS Connectivity", "FAIL", {"error": str(e)}, time.time() - test_start)

## Test 3: External API Access

In [ ]:
# Test external API (DNS API)
test_start = time.time()
try:
    import urllib.request, json
    response = urllib.request.urlopen("https://dns.google/resolve?name=example.com", timeout=5)
    data = json.loads(response.read().decode())
    log_test("External API Access", "PASS" if "Answer" in data else "FAIL", {"api": "dns.google"}, time.time() - test_start)
except Exception as e:
    log_test("External API Access", "UNKNOWN", {"error": str(e)}, time.time() - test_start)

## Summary

In [ ]:
results["summary"] = {
    "total_tests": len(results["tests"]),
    "passed": sum(1 for t in results["tests"] if t["result"] == "PASS"),
    "verdict": "PASS" if sum(1 for t in results["tests"] if t["result"] == "PASS") >= 2 else "UNKNOWN"
}

Path("/kaggle/working/discovery_02_network_probe_results.json").write_text(json.dumps(results, indent=2))
print(f"\n✅ Report saved")